Attention has undoubtedly led to tremendous advances in recent years. However, this mechanism is computationally demanding, requiring quadratic time and space complexity. As a result, researchers proposed various approximations and alternatives in the early days when attention was still computed as a single operation, refered as "batch" attention in this post. 

With the growing necessity to compute attention faster and over longer context lengths, a simple trick has far surpassed all other attempts at optimization: computing attention in an online fashion. I will call this as "online" attention. This approach processes attention piecewise, using only portions of keys, values, and queries at any given moment. It's both simple and accurate, involving no approximations.

Additionally, the online nature of attention may lead to more stable results, as the impact of larger exponents is felt only through accumulated sums. In contrast, batch attention computation can cause logits with low values (below -11.09 for FP8, when exp(-11.09) becomes smaller than the lowest representable number) to immediately drop to zero, potentially losing the contribution of corresponding value vectors. Whether this is beneficial or detrimental remains an open question, if you have insights on this, I'd love to hear them!


In this post, I will try to deconstruct how attention is computed by reading keys, values, and queries in a streaming fashion. 
To avoid memory overheads from storing intermediate representations such as score matrix we will also derive gradients through attention in an online fashion, i.e., by reading only parts of keys, values, and queries. 
I will be using code to demonstrate these concepts in practice. 

This is the mechanism behing [FlashAttention](https://github.com/Dao-AILab/flash-attention) (computing attention on a single GPU core by fusing operations and managing memory in a clever way) as well as [RingAttention](https://arxiv.org/abs/2310.01889), computing attention by cleverly overlapping partial computation of attention with communication of keys and values across GPUs. 

## Forward Pass

We have $Q \in \mathcal{R}^{N \times d}$, $K \in \mathcal{R}^{M \times d}$, and $V \in \mathcal{R}^{M \times d}$.

We want to compute 
$$
\text{Attn}(Q, K, V) = \text{Softmax}(\tau QK^T)V
$$

Let's break it down 

$$
S = QK^T \\
P = \text{Softmax}(\tau S) \\
O = PV
$$


In [25]:
A.sum(axis=-1).shape

(10, 100)

In [28]:
import numpy as np

B = 10
T = 100
d = 32
tau = 1. / d

Q = np.random.randn(B, T, d)
K = np.random.randn(B, T, d)
V = np.random.randn(B, T, d)

S = Q @ K.transpose((0, 2, 1))
print(f"S shape: {S.shape}")

m = S.max(axis=-1, keepdims=True)
print(f"m shape: {m.shape}")

A = np.exp(tau * S - m) 
A = A / A.sum(axis=-1, keepdims=True)
print(f"A shape: {A.shape}")

O = A @ V
print(f"Output shape: {O.shape}")


S shape: (10, 100, 100)
m shape: (10, 100, 1)
A shape: (10, 100, 100)
Output shape: (10, 100, 32)


## Forward pass in online fashion

We want to compute the above in online fashion, where only the blocks of Q, K, and V are available at a time. This might be due to memory constraints of GPU (e.g., FlashAttention) or due to the harware memoery constraints (e.g., Ring Attention). 

Now, note that the full computation can be written as 

$$
S_{ij} = Q_i^TK_j \\ 

P_{ij} = \frac{\exp (\tau S_{ij})}{\sum_{j} \exp (\tau S_{ij})} \\

O_{i, :} = \sum_{j}P_{i, j} \times V_{j, :}
$$

Thus, we want to compute 

$$
O_{i, :} = \frac{\sum_{j}\exp (\tau Q_i^TK_j)  \times V_{j, :}}{\sum_{j} \exp (\tau Q_i^TK_j)}
$$

To avoid overflow resulting from large exponents, we normalize the expoenentials by subtracting the maximum of scores from their exponents. 

$$
m_i = \max_{j}Q_i^TK_j \\

O_{i, :} = \frac{\sum_{j}\exp (\tau Q_i^TK_j - m_i)  \times V_{j, :}}{\sum_{j} \exp (\tau Q_i^TK_j  - m_i)}
$$


Let's denote the block of queries by $B_q$, and the block of keys and values as $B_k$.
Thus, we have $Q_{B_q} \in \mathcal{R}^{B_q \times d}$, and $K_{B_k} \in \mathcal{R}^{B_k \times d}$, and $V \in \mathcal{R}^{B_k \times d}$, where $B_q < N$ and $B_k < M$.
To do the above compuation, when only blocks of $Q, K$, and $V$ are available, we need to break it down in blocks. 


For convenience, imagine $B_q$ and $B_k$ to be $1$. For a given query, when we have read the $t$-th key and value, we can upadte the numerator, denominator, and m_i as follows:

$$
m_i^t = \max (m_i^{t-1}, Q_i^TK_t) \in \mathcal{R}\\ 

n_i^t = n_i^{t-1} \times \exp (m_i^{t-1} - m_i^t) + \exp (\tau Q_i^TK_t - m_i^{t}) \times V_t \in \mathcal{R}^d \\ 

d_i^t = d_i^{t-1} \times \exp (m_i^{t-1} - m_i^t) + \exp (\tau Q_i^TK_t - m_i^{t}) \in \mathcal{R} \\

$$

Note that the correction term $\exp (m_i^{t-1} - m_i^t)$ accounts for the update in the maximum for normalizing the exponents. 

At the end of the $M$-th reading of the keys, we will have

$$
O_{i, :} = \frac{n_i^{M}}{d_i^{M}}
$$

Thus, we can compute the attention in an online fashion, where we only need to keep the blocks of Query, Keys, and Values in memory and update the output matrix iteratively.



In [87]:
B, T, d = 10, 100, 32
tau = 1./d
Q = np.random.randn(B, T, d)
K = np.random.randn(B, T, d)
V = np.random.randn(B, T, d)


block_size = 10
m = np.zeros((B, T, 1))
m_new = np.zeros((B, T, 1))
n = np.empty((B, T, d))
d = np.empty((B, T, 1))
for col_start in range(0, T, block_size):
    col_end = col_start + block_size 
    K_b, V_b = K[:, col_start:col_end], V[:, col_start:col_end]

    for row_start in range(0, T, block_size):
        row_end = row_start + block_size 
        Q_b = Q[:, row_start: row_end]

        S_b = Q_b @ K_b.transpose((0, 2, 1))

        # compute new maximums
        m_b = S_b.max(axis=-1, keepdims=True)
        m_new[:, row_start:row_end] = np.where(m[:, row_start:row_end] >  m_b, m[:, row_start:row_end], m_b)

        # numerator
        n[:, row_start:row_end] = n[:, row_start:row_end] * np.exp(m_new[:, row_start:row_end] - m[:, row_start:row_end])
        n[:, row_start:row_end] += np.exp(tau*S_b - m_new[:, row_start:row_end]) @ V_b

        # denominator 
        d[:, row_start:row_end] = d[:, row_start:row_end] * np.exp(m_new[:, row_start:row_end] - m[:, row_start:row_end])
        d[:, row_start:row_end] += np.exp(tau*S_b - m_new[:, row_start:row_end]).sum(axis=-1, keepdims=True)

        # update m
        m = m_new

        
O_online = n/d

# print(O.max(), O.min(), O)

# fwd pass
S = Q @ K.transpose((0, 2, 1))
m = S.max(axis=-1, keepdims=True)
A = np.exp(tau * S - m) 
A = A / A.sum(axis=-1, keepdims=True)

O_batch = A @ V

print(np.allclose(O_online, O_batch))


False


In [88]:
O_online[0, :10, :2], O_batch[0, :10, :2]

(array([[ 1.13887198e+03,  1.34140959e+03],
        [ 4.41414906e-01, -1.66958785e-02],
        [ 2.79628619e-01, -6.35236233e-02],
        [ 7.23948692e-01,  2.46914251e-01],
        [ 1.01731364e+05,  7.20774792e+04],
        [ 2.79770810e+03,  2.60216726e+03],
        [ 6.52989473e-01,  8.13012789e-02],
        [ 5.86165284e-01,  1.15468689e-01],
        [ 9.10863749e+00,  4.98959803e+00],
        [ 7.31667172e-01,  4.19192166e-01]]),
 array([[ 0.06312189, -0.20470696],
        [ 0.06694846, -0.20299662],
        [ 0.03964471, -0.19253645],
        [ 0.04215725, -0.20528238],
        [ 0.06692323, -0.19983785],
        [ 0.07035043, -0.20136996],
        [ 0.07016064, -0.19919389],
        [ 0.08174251, -0.20654894],
        [ 0.03324124, -0.20901154],
        [ 0.06515493, -0.20280062]]))

**Note:** Online method is more stable as at every iteration when it computes the exponential, it keeps the value in safe range, thus the exponentials never become too negative to go beyond the representation regime. 

## Backward Pass

Let's simplify the above notations a bit. At the end of the above forward computation, we have access to the sum in the denominator, $d_i = d^{M}_i$, and the $m_i = \max_j Q_i^TK_j$. 

During the backward pass, we are interested in computing $\frac{\partial L}{\partial Q}$, $\frac{\partial L}{\partial K}$, $\frac{\partial L}{\partial V}$, given $\frac{\partial L}{\partial O}$, where $L \in \mathcal{R}$ is the loss function. For simplicity, we will avoid typing $\partial L$ so that $\partial O = \frac{\partial L}{\partial O}$ and so on. 

Note,
$$
\partial O \in \mathcal{R}^{N \times d}, \quad \partial Q \in \mathcal{R}^{N \times d}, \quad  \partial K \in \mathcal{R}^{M \times d}, \quad  \partial V \in \mathcal{R}^{M \times d}
$$

$$
O = PV  \in \mathcal{R}^{N \times d}\\

\partial V = P^T \partial O \in \mathcal{R}^{M \times d}, \qquad \partial P = \partial O V^T \in \mathcal{R}^{N \times M} \\
$$

Since each $P_{i, :}$ is a result of the softmax on the $i$-th row of $S$, while taking the derivative, we need to treat them together. 

Specifically,

$$
\mathbf{y} = \text{Softmax}(\mathbf{x}) \in \mathcal{R}^d \\

\partial \mathbf{x} = (\text{diag}(\mathbf{y}) - \mathbf{y}\mathbf{y}^T) \times \partial \mathbf{y}
$$

Above follows from the following observation - 

$$
\frac{\partial \mathbf{y}_{i}}{\partial \mathbf{x}_j} = \sigma_i \sigma_j, \quad \text{if i} \neq {j} \\
= \sigma_i (1 - \sigma_i), \text{otherwise}
$$

Applying it to the $i$-th row of $P$, we have

$$
\partial S_{i, :} = (\text{diag}(\mathbf{P_{i, :}}) - \mathbf{P_{i, :}}\mathbf{P_{i, :}}^T) \times \partial \mathbf{P_{i, :}} \in \mathcal{R}^{M},
$$

where I have assumed $\mathbf{P_{i, :}} \in \mathcal{R}^M$, a column vector even though it represents a row in the matrix $P$ 

Given $\partial S_{ij}$, we can proceed in the similar fashion to compute $\partial Q$ and $\partial K$ since $S = QK^T$ is the matrix multiplication.

$$
\partial Q = \partial S K \in \mathcal{R}^{N \times d}, \qquad \partial K^T = Q^T \partial S \in \mathcal{R}^{d \times M}

$$


If you are an expert in matrix calculus, above is easy. If you are not, checkout the appendix on how to arrive at the above formulas. 

In [ ]:
# backward pass in batch updates

### Online pass

We have $\partial O \in \mathcal{R}^{N \times d}$, $Q \in \mathcal{R}^{N \times d}$, $K \in \mathcal{R}^{M \times d}$, and $V \in \mathcal{R}^{M \times d}$.
We also have $\mathbf{d} \in \mathcal{R}^{N}$ and $\mathbf{m} \in \mathcal{R}^{N}$.

We need to compute this in the online fashion:

$$
\partial V = P^T \partial O \in \mathcal{R}^{M \times d}, \\

\partial Q = \partial S K \in \mathcal{R}^{N \times d}, \\

\partial K^T = Q^T \partial S \in \mathcal{R}^{d \times M}
$$


#### Computing $\partial V = P^T \partial O \in \mathcal{R}^{M \times d}$

$$
\partial V_{j, :} = \sum_{i=1}^N P_{ij} \times \partial O_{i, :} \\
\partial V_{j, :} = \sum_{i=1}^N \frac{exp(\tau Q_i^TK_j - m_i)}{d_i}\times \partial O_{i, :}

$$


#### Computing $\partial Q$

Let's see if we can simplify $\partial S_{i:}$

$$
\partial S_{i, :} = P_{i:} \cdot \partial P_{i:} - P_{i:} \times P_{i:}^T\partial P_{i:}
$$

Remember, from above, $\partial P_{ij} = \partial O_{i}^TV_j$

Now, observe that $ P_{i:}^T\partial P_{i:}$ can be written as follows

$$
\sum_{j} P_{ij} \times \partial P_{ij} = \sum_{j} P_{ij} \times \partial O_i^TV_j = \partial O_i^T(\sum_{j} P_{ij} \times V_j) = \partial O_i^TO_i
$$

This can be computed in the beginning because we are given $\partial O_i$ and $O_i$.


Thus, 

$$
\partial S_{ij} = P_{ij} \times \partial P_{ij} - P_{ij} \times \partial O_i^T O_i = P_{ij} (\partial P_{ij} - \partial O_i^T O_i)
$$

Finally, 

Let's express $\partial Q$ in terms of $\partial S_{ij}$ 

$$
\partial Q_{i:} = \sum_{j=1}^{M} \partial S_{ij} \times K_j \\ 

\partial Q_{i:} = \sum_{j=1}^{M} P_{ij} \times (\partial P_{ij} - \partial O_i^T O_i) \times K_j \\ 

\partial Q_{i:} = \sum_{j=1}^{M} \frac{exp(\tau Q_i^TK_j - m_i)}{D_i} \times (\partial O_i^TV_j - \partial O_i^T O_i) \times K_j \\ 
$$


### Computing $\partial K$
Let's simplify the above expression, and rewrite it as $\partial K = \partial S^TQ$

$$

\partial K_{j:} = \sum_{i=1}^{N} \partial S_{ij} Q_i \\
\partial K_{j:} = \sum_{i=1}^{N} P_{ij} \times (\partial P_{ij} - \partial O_i^T O_i) Q_i \\
\partial K_{j:} = \sum_{i=1}^{N} \frac{exp(\tau Q_i^TK_j - m_i)}{D_i} \times (\partial O_i^TV_j - \partial O_i^T O_i) \times Q_i \\
$$



## Appendix 

Here we will build the intuition to perform the following,

$$
O = PV  \in \mathcal{R}^{N \times d}\\

\partial V = P^T \partial O \in \mathcal{R}^{M \times d}, \qquad \partial P = \partial O V^T \in \mathcal{R}^{N \times M} \\
$$

### Computing $\Delta L$: Pushforward and Pullback

Let's say $\delta O$ represents the perturbation in $O$ due to the changes in its inputs - this is called the **pushforward** (or differential) of the mapping $f: (P,V) \mapsto O = PV$. 

At the same time, we have the **pullback** of the differential form $dL$ along the same mapping, which gives us $\frac{\partial L}{\partial O}$ - the gradient of $L$ with respect to $O$.

The change in $L$ due to the perturbation $\delta O$ is then given by the inner product between the gradient and the perturbation:

$$
\langle \frac{\partial L}{\partial O}, \delta O \rangle = 
\begin{cases}
\text{trace}\left(\frac{\partial L}{\partial O}^T \delta O\right) & \text{if } \frac{\partial L}{\partial O} \text{ is a matrix} \\
\left(\frac{\partial L}{\partial O}\right)^T \delta O & \text{if } \frac{\partial L}{\partial O} \text{ is a vector}
\end{cases}
$$


### Computing $\partial V$

Let's focus on the $j$-th row of $V \in \mathcal{R}^{M \times d}$.

In the forward computation we have, using the row form of matrix multiplication,  
$$
O_{i, :} = \sum_j P_{ij} \times V_{j, :}
$$

Thus, each $V_{j, :}$ contributes to $O_{i, :}$ through $P_{ij}$.

Let's see how much will $O_{i, :}$ change, if we perturb $V_{j, :}$ by $\delta V_{j, :}$

$$
\delta O_{i, :} = P_{ij} \times \delta V_{j, :}, \qquad \forall i \in \{1, 2, ..., N\}
$$

Thus, $\partial V_{j, :}$ can be written as 

$$
\partial L = \sum_{i = 1}^{N} \frac{\partial L}{\partial O_{i, :}} \times P_{ij} \times \delta V_{j, :} \\

\frac{\partial L}{\partial V_{j, :}} = \sum_{i = 1}^{N} \frac{\partial L}{\partial O_{i, :}} \times P_{ij} \\

\partial V_{j, :} = P_{:, j}^T \partial O
$$

Extending it to all rows in $V$,

$$
\partial V = P^T\partial O
$$

### Computing $\partial P_{ij}$

Using column-view of matrix multiplication, 

$$
\delta O_{i,:}  = \delta P_{ij} \times V_{j,:}
$$

Thereby, we can calculate $\delta L$ as ($\frac{\partial L}{\partial O_{i, :}} \in \mathcal{R}^{d}$, $V_{j, :} \in \mathcal{R}^d$)

$$
\delta L = \frac{\partial L}{\partial O_{i, :}}^T (\delta P_{ij} \times V_{j,:}) \\ 

\delta L = \frac{\partial L}{\partial O_{i, :}}^T (V_{j,:}) \delta P_{ij} \\ 

 \frac{\partial L}{\partial P_{ij}} = \partial P_{ij} = \partial O_{i, :}^TV_{j, :}
$$


Completing the picture, we get

$$
\partial P = \partial OV^T
$$